# Token Representation Geometry

Geometric properties of token representations:
clustering, spread, velocity, and inter-token distances.

## Why This Matters

Token representation geometry tracks how individual token representations evolve through the network. Understanding token-level dynamics reveals how context is integrated, when predictions form, and how different positions interact to produce the output.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.token_representation_geometry import (
    representation_clustering, representation_spread,
    representation_velocity, inter_token_distances,
    representation_geometry_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Representation Clustering

Are token representations converging to similar points?

In [ ]:
for layer in range(4):
    result = representation_clustering(model, tokens, layer=layer)
    flag = ' [CLUSTERED]' if result['is_clustered'] else ''
    print(f"  Layer {layer}: sim={result['mean_pairwise_similarity']:.4f}{flag}")

## Representation Spread

Effective dimensionality of the representation manifold.

In [ ]:
for layer in range(4):
    result = representation_spread(model, tokens, layer=layer)
    flag = ' [LOW RANK]' if result['is_low_rank'] else ''
    print(f"  Layer {layer}: eff_rank={result['effective_rank']:.2f}, "
          f"top_sv_frac={result['top_sv_fraction']:.4f}{flag}")

## Representation Velocity

How fast does the representation change between layers?

In [ ]:
result = representation_velocity(model, tokens, position=-1)
flag = ' [DECELERATING]' if result['is_decelerating'] else ''
print(f"Mean velocity: {result['mean_velocity']:.4f}{flag}")
for p in result['per_layer']:
    bar = '█' * int(p['velocity'] * 20)
    print(f"  Layer {p['layer']}: v={p['velocity']:.4f}, rel={p['relative_velocity']:.4f} {bar}")

## Inter-Token Distances

Pairwise Euclidean distances between token representations.

In [ ]:
for layer in range(4):
    result = inter_token_distances(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean={result['mean_distance']:.4f}, "
          f"min={result['min_distance']:.4f}, max={result['max_distance']:.4f}")

## Geometry Summary

In [ ]:
result = representation_geometry_summary(model, tokens)
for p in result['per_layer']:
    flag = ' [CLUSTERED]' if p['is_clustered'] else ''
    print(f"  Layer {p['layer']}: sim={p['mean_similarity']:.4f}, "
          f"eff_rank={p['effective_rank']:.2f}{flag}")